## **IMDB Sentiment Analysis using NLP Pipeline & ML Models**

# **Import Libraries**

In [134]:
import pandas as pd
import numpy as np

# NLP
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# **Load data**

In [135]:
import pandas as pd

df = pd.read_csv(
    "IMDB Dataset.csv",
    engine='python',
    sep=',',                # separator
    encoding='utf-8',
    on_bad_lines='skip'     # skip broken rows
)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [136]:
df = df.dropna()

In [137]:
#df = pd.read_csv("IMDB Dataset.csv",quoting=3, encoding='utf-8')
#df.head()

# **Basic Checks**

In [138]:
print(df.shape)

(38816, 2)


In [139]:
print(df.columns)

Index(['review', 'sentiment'], dtype='object')


In [140]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [141]:
df.tail()

,review,sentiment
38811,If you ever watched the Dukes of Hazard you kn...,negative
38812,This film was basically Velvet Goldmine if the...,negative
38813,The centerpiece of Lackawanna Blues is the cha...,positive
38814,"I really hope the makers of these ""movies"" rea...",negative
38815,This movie is like Happiness meets Lost in Tra...,positive


In [142]:
print(df.columns)

Index(['review', 'sentiment'], dtype='object')


In [143]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38816 entries, 0 to 38815
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     38816 non-null  object
 1   sentiment  38816 non-null  object
dtypes: object(2)
memory usage: 606.6+ KB


In [144]:
print(df['sentiment'].value_counts())

sentiment
positive    19419
negative    19397
Name: count, dtype: int64


# **NLP Preprocessing**

### NLP Preprocessing Steps
- Converted text to lowercase
- Removed punctuation
- Removed stopwords
- Tokenization
- Stemming/Lemmatization
- Removed special characters and URLs

In [145]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# **Create preprocessing function**

In [146]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

# **Apply preprocessing**

In [147]:
print(df.columns)


Index(['review', 'sentiment'], dtype='object')


In [148]:
df['clean_text'] = df['review'].apply(preprocess_text)

df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod hook right ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic famili littl boy jake think zombi closet...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


# **Feature Engineering**

In [149]:
y = df['sentiment']

In [150]:
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text'])
X_bow.shape

(38816, 5000)

# **TF-IDF**

In [151]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text'])
X_tfidf.shape

(38816, 5000)

# **Train-Test-Split**

In [152]:
print(X_bow.shape[0])
print(len(y))

38816
38816


In [153]:
print(X_bow.shape)
print(len(y))

(38816, 5000)
38816


In [154]:
df = df.dropna()

X = df['review']
y = df['sentiment']

In [155]:
#Split bow

X_train, X_test, y_train, y_test= train_test_split(X_bow, y, test_size=0.2, random_state=42)

In [156]:
#Train Model
lr_bow = LogisticRegression(max_iter=2000)
lr_bow.fit(X_train, y_train)

LogisticRegression(max_iter=2000)

In [157]:
# Logistic Regression with BoW
#predict
y_pred_lr_bow = lr_bow.predict(X_test)


In [158]:
#Evaluate
evaluate(y_test, y_pred_lr_bow)

Accuracy: 0.8624420401854714
Precision: 0.8624708056593431
Recall: 0.8624420401854714
F1 Score: 0.862443546314209
----------------------------------------


# **Model Building**

In [159]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=50, max_depth=6, learning_rate=0.1, random_state=42)
}

**1)Logistic Regression**

In [160]:
y = df['sentiment']

x_train, x_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [161]:
lr = LogisticRegression()
lr.fit(x_train, y_train)

y_pred_lr = lr.predict(x_test)

**2)Naive Bayes**

In [162]:
nb = MultinomialNB()
nb.fit(x_train, y_train)

y_pred_nb = nb.predict(x_test)

**Decision Tree**

In [163]:
dt = DecisionTreeClassifier()
dt.fit(x_train, y_train)

y_pred_dt = dt.predict(x_test)

# **Model Evaluation**

In [164]:
### Evaluation Metrics Accuracy,Precision,Recall,F1 Score
def evaluate(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("-"*40)

 **Evaluate all models**

# **Comparision Table**

In [165]:
comparison_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.884080,0.884324,0.884080,0.884073
1,Naive Bayes,0.855100,0.855237,0.855100,0.855097
2,Decision Tree,0.714451,0.714653,0.714451,0.714426


In [166]:
results = []

models = {
    "Logistic Regression": y_pred_lr,
    "Naive Bayes": y_pred_nb,
    "Decision Tree": y_pred_dt
}

for name, pred in models.items():
    results.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred, average='weighted'),
        recall_score(y_test, pred, average='weighted'),
        f1_score(y_test, pred, average='weighted')
    ])

comparison_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score"
])

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.884080,0.884324,0.884080,0.884073
1,Naive Bayes,0.855100,0.855237,0.855100,0.855097
2,Decision Tree,0.720118,0.720207,0.720118,0.720115


# **Best Model Code**

In [167]:
best_model = comparison_df.sort_values(by="F1 Score", ascending=False).iloc[0]
print(best_model)

Model        Logistic Regression
Accuracy                 0.88408
Precision               0.884324
Recall                   0.88408
F1 Score                0.884073
Name: 0, dtype: object


# **Final Insight**


- Preprocessing improved text quality and model performance
- TF-IDF performed better than Bag of Words in most cases
- Logistic Regression / XGBoost achieved the highest F1-score
- Naive Bayes was faster but slightly less accurate
- Decision Tree showed overfitting tendency

### Trade-offs
- Simpler models are faster but less accurate
- Complex models (XGBoost) give better performance but take more time

### Real-world Application
- Movie review sentiment analysis
- Customer feedback analysis
- Product review classification